In [1]:
!pip install sentence-transformers pandas openpyxl

In [2]:
from google.colab import files
uploaded = files.upload()

Saving RESUME ANALYSIS LLM PROJECT.xlsx to RESUME ANALYSIS LLM PROJECT.xlsx


In [19]:
import pandas as pd
from sentence_transformers import SentenceTransformer

SRC = "RESUME ANALYSIS LLM PROJECT.xlsx"   # matches your uploaded filename exactly
OUT = "RESUME_ANALYSIS_LLM_PROJECT_Phase4_Embeddings_REAL.xlsx"

# --- 1. Load chunked data from your Excel ---
chunks_df = pd.read_excel(SRC, sheet_name="chunked data")

# --- 2. Load the embedding model (downloads once, ~80MB) ---
model = SentenceTransformer("all-MiniLM-L6-v2")

# --- 3. Generate embeddings for every chunk ---
texts = chunks_df["Chunk_Text"].tolist()
embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

# --- 4. Build output dataframe ---
emb_df = pd.DataFrame(embeddings, columns=[f"dim_{i+1}" for i in range(embeddings.shape[1])])
emb_df.insert(0, "Chunk_ID", chunks_df["Chunk_ID"].values)
emb_df.insert(1, "Candidate_ID", chunks_df["Candidate_ID"].values)
emb_df.insert(2, "Embedding_Model", "all-MiniLM-L6-v2")
emb_df.insert(3, "Embedding_Dim", embeddings.shape[1])

# --- 5. Save to Excel ---
with pd.ExcelWriter(OUT, engine="openpyxl") as writer:
    emb_df.to_excel(writer, sheet_name="Chunk_Embeddings", index=False)
    chunks_df.to_excel(writer, sheet_name="Chunk_Text_Reference", index=False)

print(f"Done! Saved {len(emb_df)} embeddings of dimension {embeddings.shape[1]} to {OUT}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Done! Saved 1000 embeddings of dimension 384 to RESUME_ANALYSIS_LLM_PROJECT_Phase4_Embeddings_REAL.xlsx


In [20]:
from google.colab import files
files.download("RESUME_ANALYSIS_LLM_PROJECT_Phase4_Embeddings_REAL.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found e

In [22]:
import pandas as pd
import chromadb

# Load the REAL embeddings file you just downloaded/generated
emb_df = pd.read_excel("RESUME_ANALYSIS_LLM_PROJECT_Phase4_Embeddings_REAL.xlsx", sheet_name="Chunk_Embeddings")
ref_df = pd.read_excel("RESUME_ANALYSIS_LLM_PROJECT_Phase4_Embeddings_REAL.xlsx", sheet_name="Chunk_Text_Reference")

# Extract the 384 number-columns as vectors
dim_cols = [c for c in emb_df.columns if c.startswith("dim_")]
embeddings = emb_df[dim_cols].values.tolist()

# Create a persistent ChromaDB store (saved to disk, not lost when Colab restarts... within the session)
client = chromadb.PersistentClient(path="./resume_vector_db")
collection = client.get_or_create_collection(name="resume_chunks")

# Add all chunks with their embeddings, text, and metadata
collection.add(
    ids=emb_df["Chunk_ID"].tolist(),
    embeddings=embeddings,
    documents=ref_df["Chunk_Text"].tolist(),
    metadatas=[{"Candidate_ID": cid} for cid in emb_df["Candidate_ID"].tolist()],
)

print(f"Stored {collection.count()} chunks in ChromaDB")

Stored 1000 chunks in ChromaDB


In [23]:
results = collection.query(
    query_texts=["candidate skilled in Python and machine learning"],
    n_results=3
)
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(meta["Candidate_ID"], "->", doc[:100], "...")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:10<00:00, 8.05MiB/s]


CAND0295 -> Ph.D graduate in Mathematics with 5.8 years of experience, applying for the Backend Developer positi ...
CAND0352 -> is a MCA graduate in Mechanical Engineering with 2.9 years of experience, applying for the Full Stac ...
CAND0447 -> a B.E. graduate in Artificial Intelligence with 5.0 years of experience, applying for the Full Stack ...


In [24]:
import shutil
from google.colab import files

shutil.make_archive("resume_vector_db", "zip", "resume_vector_db")
files.download("resume_vector_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import os
print(os.path.exists("resume_vector_db"))

True


In [28]:
import shutil
from google.colab import files

shutil.make_archive("resume_vector_db", "zip", "resume_vector_db")
files.download("resume_vector_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
!pip install transformers

In [37]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5 model and tokenizer directly
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def ask_rag(question, n_results=3):
    # Step 1: Retrieve relevant chunks from ChromaDB (from Phase 5)
    results = collection.query(query_texts=[question], n_results=n_results)
    retrieved_chunks = results["documents"][0]
    candidate_ids = [m["Candidate_ID"] for m in results["metadatas"][0]]
    candidate_ids = list(dict.fromkeys(candidate_ids))  # removes duplicates, keeps order

    # Step 2: Build a more directive prompt to get a fuller answer
    context = "\n".join(retrieved_chunks)
    prompt = f"""You are a recruitment assistant. Based on the candidate information below, write 2-3 sentences explaining which candidates best match the question and why.

Candidate Information:
{context}

Question: {question}

Write a detailed answer:"""

    # Step 3: Generate the answer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_length=200)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response, candidate_ids, retrieved_chunks

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [38]:
question = "Who are the best candidates for a Python and machine learning role?"
answer, ids, chunks = ask_rag(question)

print("QUESTION:", question)
print("\nANSWER:", answer)
print("\nRETRIEVED CANDIDATES:", ids)

QUESTION: Who are the best candidates for a Python and machine learning role?

ANSWER: Sai Khan is a B.E. graduate in Artificial Intelligence with 5.0 years of experience.

RETRIEVED CANDIDATES: ['CAND0447', 'CAND0163']


In [41]:
def rag_pipeline(question, n_results=3, verbose=False):
    """
    Complete RAG pipeline for AI Resume Screening.
    Input: a recruiter's question (string)
    Output: a generated answer + supporting candidate IDs + retrieved text
    """

    # STAGE: Retriever — search ChromaDB for the most relevant chunks
    results = collection.query(query_texts=[question], n_results=n_results)
    retrieved_chunks = results["documents"][0]
    candidate_ids = [m["Candidate_ID"] for m in results["metadatas"][0]]
    candidate_ids = list(dict.fromkeys(candidate_ids))  # dedupe

    # STAGE: Prompt Template — combine retrieved context with the question
    context = "\n".join(retrieved_chunks)
    prompt = f"""You are a recruitment assistant. Based on the candidate information below, write 2-3 sentences explaining which candidates best match the question and why.

Candidate Information:
{context}

Question: {question}

Write a detailed answer:"""

    # STAGE: LLM — generate the final answer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_length=200)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if verbose:
        print("Retrieved chunks:", len(retrieved_chunks))
        print("Candidates:", candidate_ids)

    return {
        "question": question,
        "answer": answer,
        "candidate_ids": candidate_ids,
        "retrieved_chunks": retrieved_chunks,
    }

In [42]:
result = rag_pipeline("Who are the best candidates for a Network Engineer role with certifications?")
print(result["answer"])
print(result["candidate_ids"])

The candidates are: David Wang, Matthew Anderson, and Hassan Anderson.
['CAND0260', 'CAND0221', 'CAND0061']
